In [ ]:
import torch

# 1. (b). Fine-tuning

The task that you will be working on here is to classify whether a given English sentence is grammatically correct or not. So, this is a task of text classification. You will be working with the CoLA dataset about which you will get know in one of the other slides below.

##*NOTE: Most of the code below is already completed for you. You need to complete the missing code in the cells. Also, do not miss any question that is asked in the text cells below. You need to complete the code in this Notebook and submit it. For the text answers, you may answer it here in the text cells, or you may answer them in the PDF along with other answers. If you do not submit the notebook, no marks will be awarded!*

First we will install some libraries that we need here.

In [ ]:
!pip install wget
!pip install transformers

Next, we want to import all the important libraries we need.

In [ ]:
import os
import wget
import random
import numpy as np
import pandas as pd

import torch
from torch.utils.data import (TensorDataset, 
                              DataLoader, 
                              RandomSampler, 
                              SequentialSampler)

from transformers import (BertTokenizer, 
                          BertForSequenceClassification,
                          AdamW,
                          BertConfig,
                          get_linear_schedule_with_warmup)

# Don't worry, we are not using Tensorflow here.
# Just using a function from the Keras library
from tensorflow.keras.utils import pad_sequences

from sklearn.model_selection import train_test_split

We now assign the device on which we want our model to train. Preferbly a GPU, if not, then your CPU.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print('There are %d GPU(s) available.' % torch.cuda.device_count())
    print('We will use the GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

# TODO: Download the dataset

Here we download our data. We use the CoLA dataset.

The Corpus of Linguistic Acceptability (CoLA) dataset [here](https://nyu-mll.github.io/CoLA/cola_public_1.1.zip) consists of 10657 sentences from 23 linguistics publications, expertly annotated for acceptability (grammaticality) by their original authors.

In [ ]:
print('Downloading dataset...')

# TODO:
# Download the file



# TODO:
# Unzip the dataset (if you have not already)



# TODO: Load the dataset

Here we load the data.

In [ ]:
# TODO:
# Load the dataset (in_domain_train.tsv) into a pandas dataframe.
# use the tab delimiter, and the columns ['sentence_source', 'label', 'label_notes', 'sentence']

df = 

print('Number of training sentences: {:,}\n'.format(df.shape[0]))

# TODO:
# Display top 10 rows from the data



In [ ]:
# TODO:
# Store the sentences and labels of the data in the variables

sentences = 
labels = 

# TODO: Text Preprocessing

Here we initialize the BERT tokenizer. You need to complete the code so that the tokenizer is loaded. Next, run the piece of code below and it should work properly if you have loaded the tokenizer without errors.

In [ ]:
MODEL_TOKENIZER_NAME = 'bert-base-uncased'

# TODO:
# Initialise the BertTokenizer here using the MODEL_TOKENIZER_NAME above and make sure the text is lowercase
# Refer: https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertTokenizer

print('Loading BERT tokenizer...')
tokenizer = 

Here we apply the tokenization to the data and encode the tokens to ids.


In [ ]:
input_ids = []

for sent in sentences:

    # TODO:
    # Use the tokenizer to tokenize each sentence 'sent'
    # Remember to add the special [CLS] and [SEP] tokens
    # If you are confused, refer the link in the previous cell

    encoded_sent = 
    
    input_ids.append(encoded_sent)

Here we first find the max sequence length of our data and then apply padding to the data.


In [ ]:
# TODO:
# Calculate the max sequence length of the data (input_ids)

print('Max sentence length: ', max(_____________________))

# Set the maximum sequence length.
# Although the max length is 47, we choose 64 as the max length here.
MAX_LEN = 64

print('\nPadding all sentences in data ...')

input_ids = pad_sequences(input_ids, maxlen=MAX_LEN, dtype="long", 
                          value=0, truncating="post", padding="post")

print('\nDone.')

Here we create the attention mask for our sentences.

If a token ID is 0, then its padding. Set mask to 0.

If a token ID is not 0, then its a real token. Set mask to 1.

In [ ]:
# Create attention masks
attention_masks = []

for sent in input_ids:
    
    # TODO:
    # Calculate the attention mask here using the logic in the description above
    att_mask = _____________________
    
    # Store the attention mask for this sentence.
    attention_masks.append(att_mask)

# TODO: Data split

Here we split the dataset into train and validation sets. You may use the train_test_split() function from sklearn for this. Use any random_state value so that your results are reproducible. You may use the test set size of 10%.

Remember you also need to convert the values to tensors as this is pytorch.

In [ ]:
# TODO:
# Split the data into 90% training and 10% validation

train_inputs, validation_inputs, train_labels, validation_labels = 

# TODO:
# Now split the masks in the same way

train_masks, validation_masks, _, _ = 

# TODO:
# Convert the values to the tensors here

train_inputs = 
validation_inputs = 

train_labels = 
validation_labels = 

train_masks = 
validation_masks = 

Use the PyTorch DataLoader here to load the data into batches of BATCH_SIZE

In [ ]:
# PyTorch DataLoader needs to know our batch size for training

BATCH_SIZE = 32

# TODO:
# Create the DataLoader for our training set

train_data = TensorDataset(_________)
train_sampler = RandomSampler(______)
train_dataloader = DataLoader(______)

# TODO:
# Now do the same for the validation set

validation_data = 
validation_sampler = 
validation_dataloader = 

# TODO: Your Model

Here you load the pretrained model and load it into an instance of BertForSequenceClassification. This part is already done for you.

NOTE: This takes some time to download the pretrained model (~440 MB).

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels = 2,
    output_attentions = False,
    output_hidden_states = False,
)

model.cuda()

Next we prepare the Optimizer and the learning rate scheduler. AdamW is a class from the HuggingFace library. We use this as our optimizer.

In [ ]:
# TODO:
# Initialise the optimizer here using AdamW.
# Use the parameters of your model above, and the learning rate and epsilon values given below.
# Refer: Check the AdamW library from HuggingFace

LEARNING_RATE = 2e-5
EPSILON = 1e-8
optimizer = 

epochs = 4

# Total number of training steps is number of batches * number of epochs.
total_steps = len(train_dataloader) * epochs

# TODO:
# Create the learning rate scheduler with the optimizer
# Use zero warmup steps and the total steps of training as inputs
# Check the get_linear_schedule_with_warmup function from HuggingFace

scheduler = 

Here you will define the function to find the accuracy of your predictions.

In [ ]:
# TODO:
# Function to calculate the accuracy
# You need to find how many times predictions are equal to labels, and then divide it by number of labels

def flat_accuracy(preds, labels):
  
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return ________________

Training and Validation phases. Complete the missing pieces of code, and ensure that the code runs properly with errors.

In [ ]:
# TODO:
# Set your own seed value for easy reproducibility of your results

seed_val = _________

random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

loss_values = []

######### TRAINING PHASE ###########

for epoch_i in range(epochs):
    
    print("")
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
    print('Training...')

    total_loss = 0
    model.train()

    # For each batch
    for step, batch in enumerate(train_dataloader):

        # Progress logged every 50 batches
        if step % 50 == 0 and not step == 0:

            print('  Batch {:>5,}  of  {:>5,}'.format(step, len(train_dataloader)))

        # TODO: 
        # The batch variable contains input_ids, input_masks, and labels.
        # You need to assign each of them to the 'device' for training

        b_input_ids = 
        b_input_mask = 
        b_labels = 

        # TODO:
        # Here you need to clear any previously calculated gradients
        
        

        # TODO:
        # Perform a forward pass here
        # Remember that your model will need input_ids, input_mask, and labels
        
        outputs = 
        
        # TODO:
        # The outputs variable above should now have the loss.
        # Extract the loss, and add it to the total_loss variable
        # NOTE THAT THESE ARE ALL TENSORS!!!
        
        total_loss += 

        # TODO:
        # Perform a backward pass here
        


        # The below code clips the gradients to 1.0 to prevent 
        # the "exploding gradients" problem.
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # TODO:
        # Update the parameters of the model here
        
        

        # TODO:
        # Update the learning rate here
        
        

    # TODO:
    # Calculate the average loss over the training data
    
    avg_train_loss = 
    loss_values.append(avg_train_loss)

    print("")
    print("  Average training loss: {0:.2f}".format(avg_train_loss))
        
######### VALIDATION PHASE ###########

    print("")
    print("Running Validation...")

    model.eval()

    eval_loss, eval_accuracy = 0, 0
    nb_eval_steps, nb_eval_examples = 0, 0

    for batch in validation_dataloader:
        
        batch = tuple(t.to(device) for t in batch)
        b_input_ids, b_input_mask, b_labels = batch

        with torch.no_grad():        

            # TODO:
            # Perform a forward here too, this time for validation

            outputs = 
        
        logits = outputs[0]
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()
        
        tmp_eval_accuracy = flat_accuracy(logits, label_ids)
        eval_accuracy += tmp_eval_accuracy
        nb_eval_steps += 1

    print("  Accuracy: {0:.2f}".format(eval_accuracy/nb_eval_steps))

print("")
print("Training DONE.")